# Collect QCEW total employment 

In [ ]:
from pathlib import Path

import pandas as pd
import time


# ============================================================
# COLLECT QCEW TOTAL EMPLOYMENT
# ============================================================

def collect_qcew_total_employment(year):
    """
    Collect total covered employment for statewide areas
    from the BLS QCEW API.

    Industry code 10 = total, all industries.
    Ownership code 0 = total covered employment.
    """

    url = (
        f"https://data.bls.gov/cew/data/api/"
        f"{year}/a/industry/10.csv"
    )

    print(f"Collecting {year}: {url}")

    dataframe = pd.read_csv(
        url,
        dtype={"area_fips": "string"}
    )

    # Standardize column names.
    dataframe.columns = (
        dataframe.columns
        .str.strip()
        .str.lower()
    )

    dataframe["area_fips"] = (
        dataframe["area_fips"]
        .astype("string")
        .str.strip()
        .str.zfill(5)
    )

    # Retain statewide geographic areas only.
    dataframe = dataframe.loc[
        dataframe["area_fips"].str.fullmatch(
            r"\d{2}000",
            na=False
        )
    ].copy()

    # Retain total covered ownership.
    if "own_code" in dataframe.columns:
        dataframe["own_code"] = pd.to_numeric(
            dataframe["own_code"],
            errors="coerce"
        )

        dataframe = dataframe.loc[
            dataframe["own_code"].eq(0)
        ].copy()

    dataframe["annual_avg_emplvl"] = pd.to_numeric(
        dataframe["annual_avg_emplvl"],
        errors="coerce"
    )

    result = dataframe[
        [
            "area_fips",
            "year",
            "annual_avg_emplvl"
        ]
    ].rename(
        columns={
            "annual_avg_emplvl":
                "QCEW_Total_Employment"
        }
    )

    result["year"] = year

    # Confirm one observation per area.
    if result.duplicated(
        ["area_fips", "year"]
    ).any():
        raise ValueError(
            f"Duplicate area-year rows found for {year}."
        )

    print(
        f"Collected successfully: "
        f"{len(result)} state areas"
    )

    return result

### Run the 2015–2025 loop

In [ ]:
qcew_total_frames = []
qcew_total_errors = []

for year in range(2015, 2026):

    try:
        yearly_total = collect_qcew_total_employment(
            year
        )

        qcew_total_frames.append(yearly_total)

        time.sleep(0.25)

    except Exception as error:
        qcew_total_errors.append({
            "year": year,
            "error_type": type(error).__name__,
            "error_message": str(error)
        })

        print(
            f"{year} failed: "
            f"{type(error).__name__}: {error}"
        )

### Combine and validate

In [ ]:
qcew_total_raw = pd.concat(
    qcew_total_frames,
    ignore_index=True
)

qcew_total_raw = (
    qcew_total_raw
    .sort_values(["year", "area_fips"])
    .reset_index(drop=True)
)

qcew_total_errors_df = pd.DataFrame(
    qcew_total_errors
)

print("Combined shape:", qcew_total_raw.shape)
print(
    "Years:",
    qcew_total_raw["year"].min(),
    "to",
    qcew_total_raw["year"].max()
)

display(qcew_total_raw.head())

### Check annual coverage

In [ ]:
total_employment_coverage = (
    qcew_total_raw
    .groupby("year")
    .agg(
        Rows=("area_fips", "size"),
        State_Areas=("area_fips", "nunique"),
        Missing=("QCEW_Total_Employment", lambda x: x.isna().sum())
    )
    .reset_index()
)

display(total_employment_coverage)

## Save it 

In [ ]:
RAW_BLS_DIR = Path("data") / "raw" / "bls"
RAW_BLS_DIR.mkdir(parents=True, exist_ok=True)

qcew_total_file = (
    RAW_BLS_DIR
    / "bls_qcew_total_employment_2015_2025_raw.csv"
)

qcew_total_raw.to_csv(
    qcew_total_file,
    index=False
)

qcew_total_errors_df.to_csv(
    RAW_BLS_DIR
    / "bls_qcew_total_employment_errors.csv",
    index=False
)

print("Saved:", qcew_total_file)